## Project Context

**What is the problem?**
Banks and financial institutions issue credit to customers, but a portion of
those customers fail to repay on time — this is called a "default." The
problem addressed in this project is predicting, in advance, whether a given
credit card customer is likely to default on their payment next month, based
on their demographic details, credit history, and past repayment behavior.
This is framed as a **binary classification problem** (Default = 1, No
Default = 0).

**Why is the problem important?**
Loan and credit defaults are one of the biggest sources of financial risk for
banks. If a bank cannot anticipate which customers are likely to default, it
either extends credit too freely (leading to bad debt and financial losses)
or restricts credit too conservatively (losing good customers and revenue).
Early and accurate identification of high-risk customers allows a bank to
take preventive action — adjusting credit limits, flagging accounts for
review, or offering restructured repayment plans — before an actual default
happens.

**Who will benefit from the solution?**
- **Banks and financial institutions** — better risk assessment, reduced bad
  debt, and more informed credit decisions.
- **Credit risk and collections teams** — a data-driven way to prioritize
  which accounts need proactive follow-up.
- **Customers indirectly** — banks that manage risk well can continue to
  offer credit access and fair terms rather than tightening lending broadly.
- **Regulators/auditors** — a transparent, metric-backed model supports
  responsible lending practices.

**Why is Machine Learning suitable?**
This is a pattern-recognition problem with a large number of historical,
labeled examples (past customers whose repayment outcome is already known)
and many potentially interacting features (credit limit, age, education,
repayment history, bill amounts, payment amounts, etc.). The relationship
between these features and the chance of default is not simple or linear
enough to be captured with fixed manual rules. Machine learning models can
automatically learn these complex, non-linear patterns from historical data
and generalize them to predict outcomes for new, unseen customers — something
manual rule-based systems struggle to do at scale or adapt over time as
customer behavior changes.

---


# Customer Default Prediction — Machine Learning Mini Project

**Domain:** Banking & Finance
**Problem Type:** Binary Classification
**Dataset:** Default of Credit Card Clients (UCI Machine Learning Repository)
**Target Variable:** `default payment next month` (1 = Client will default, 0 = Client will not default)

This notebook follows the complete ML workflow: Problem Understanding → Data Loading →
Data Understanding → Cleaning → Preprocessing → EDA → Model Building → Evaluation →
Model Comparison → Conclusion.

Run each cell **in order**, top to bottom.

## Block 1: Import Libraries

**What this block does:** Loads every Python library we need for the whole project —
data handling (pandas, numpy), visualization (matplotlib, seaborn), and machine
learning (scikit-learn). Doing this in one place at the top keeps the notebook clean
and makes it obvious what dependencies the project needs.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              roc_curve, auc)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

import warnings
warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

## Block 2: Load the Dataset

**What this block does:** Uploads the dataset file (`loan_dataset`) directly from
your computer using Colab's file-upload widget, then loads it into a pandas
DataFrame. This dataset should have **at least 500 records**, ideally the
*"Default of Credit Card Clients"* dataset (30,000 records, 24 features) from the
[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients).

**How to use this block:** Run the cell — a "Choose Files" button will appear.
Click it and select your `loan_dataset` file from your computer (`.csv`, `.xls`,
or `.xlsx` are all supported).

In [ ]:
from google.colab import files

files.upload()

df = pd.read_csv("loan_dataset.csv")

print("Dataset Loaded Succesfully")

## Block 3: Understand the Data

**What this block does:** Performs an initial inspection — shape (rows/columns),
column names, data types, and summary statistics. This is required for your
report's **Dataset Description** section (number of rows, columns, features,
target variable, data types).

In [ ]:
print("Shape of dataset (rows, columns):", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nData types:\n")
df.info()

In [ ]:
df.describe()

## Block 4: Clean Up Column Names & Target Variable

**What this block does:** The raw dataset has an `ID` column (not useful for
prediction, so we drop it) and a long target column name
`default payment next month`, which we rename to `default` for convenience.

In [ ]:
df = df.drop(columns=["Unnamed: 0"])
df = df.rename(columns={"Y": "default"})

print("Updated columns:\n", df.columns.tolist())
df.head()

## Block 5: Check Missing Values & Duplicate Records

**What this block does:** Confirms whether the dataset has missing values or
duplicate rows, which must be handled before modeling. This UCI dataset is
generally clean, but we check anyway — this is a mandatory reporting step
(Section 6 of the project requirements).

In [ ]:
print("Missing values per column:\n")
print(df.isnull().sum().sum(), "total missing values")

print("\nDuplicate rows:", df.duplicated().sum())

# Drop duplicates if any exist
df = df.drop_duplicates()
print("Shape after removing duplicates:", df.shape)

## Block 6: Fix Invalid Categorical Values

**What this block does:** The `EDUCATION` and `MARRIAGE` columns contain
undocumented/invalid category codes (e.g., `EDUCATION` has values 0, 5, 6 that
aren't defined in the official documentation; `MARRIAGE` has an undocumented 0).
We group these into an "Other" category (4 for EDUCATION, 3 for MARRIAGE) so the
categories are clean and meaningful before encoding.

**Why this matters:** Leaving undefined codes in the data can confuse the model
and makes visualizations harder to interpret — this is a data-cleaning step you
should explicitly justify in your report.

In [ ]:
# Promote the first row to be the new header
new_header = df.iloc[0] # Grab the first row for the headers
df = df[1:] # Take the data less the header row
df.columns = new_header # Set the header row as the DataFrame header

# Re-rename the 'default payment next month' column to 'default' after promoting headers
df = df.rename(columns={"default payment next month": "default"})

# Convert relevant columns to numeric type as they were loaded as objects
df["EDUCATION"] = pd.to_numeric(df["EDUCATION"])
df["MARRIAGE"] = pd.to_numeric(df["MARRIAGE"])

print("EDUCATION value counts before cleaning:\n", df["EDUCATION"].value_counts())
print("\nMARRIAGE value counts before cleaning:\n", df["MARRIAGE"].value_counts())

df["EDUCATION"] = df["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
df["MARRIAGE"] = df["MARRIAGE"].replace({0: 3})

print("\nEDUCATION value counts after cleaning:\n", df["EDUCATION"].value_counts())
print("\nMARRIAGE value counts after cleaning:\n", df["MARRIAGE"].value_counts())

## Block 7: EDA #1 — Target Variable Distribution

**What this block does:** Plots how many customers defaulted vs. did not default.

**Insight to write in your report:** This dataset is **imbalanced** — far fewer
customers default than don't. This is important because it means plain accuracy
can be misleading, and metrics like Recall/F1 matter more (you'll use this point
in your evaluation-metric justification and in your viva).

In [ ]:
plt.figure()
ax = sns.countplot(x="default", data=df, palette="Set2")
plt.title("Distribution of Default vs Non-Default Customers")
plt.xlabel("Default (0 = No, 1 = Yes)")
plt.ylabel("Number of Customers")

total = len(df)
for p in ax.patches:
    pct = 100 * p.get_height() / total
    ax.annotate(f"{pct:.1f}%", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()

print(df["default"].value_counts(normalize=True) * 100)

## Block 8: EDA #2 — Age Distribution

**What this block does:** Shows the age spread of customers in the dataset via a
histogram.

**Insight to write in your report:** Comment on which age range has the most
customers (typically customers are concentrated in their late 20s-40s), and
whether the dataset skews toward a particular age group.

In [ ]:
plt.figure()
sns.histplot(df["AGE"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of Customer Age")
plt.xlabel("Age")
plt.ylabel("Count")
plt.show()

## Block 9: EDA #3 — Credit Limit vs Default Status

**What this block does:** Compares the credit limit (`LIMIT_BAL`) of customers
who defaulted vs those who didn't, using a box plot.

**Insight to write in your report:** Typically, customers with **lower credit
limits show a higher tendency to default**, suggesting `LIMIT_BAL` is a useful
predictive feature (banks often assign lower limits to riskier customers to
begin with, which reinforces this pattern).

In [ ]:
plt.figure()
sns.boxplot(x="default", y="LIMIT_BAL", data=df, palette="Set3")
plt.title("Credit Limit (LIMIT_BAL) by Default Status")
plt.xlabel("Default (0 = No, 1 = Yes)")
plt.ylabel("Credit Limit (NT Dollars)")
plt.show()

## Block 10: EDA #4 — Correlation Heatmap

**What this block does:** Shows how strongly each numeric feature correlates
with every other feature, including the target.

**Insight to write in your report:** Point out which features correlate most
strongly with `default` (in this dataset, the `PAY_0`...`PAY_6` repayment-status
columns usually show the strongest correlation — meaning **past repayment
behavior is the strongest signal of future default**).

In [ ]:
plt.figure(figsize=(16, 12))
corr = df.corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False)
plt.title("Correlation Heatmap of All Features")
plt.show()

print("Top 10 features most correlated with 'default':\n")
print(corr["default"].sort_values(ascending=False).head(11))

## Block 11: EDA #5 — Default Rate by Education & Marital Status

**What this block does:** Breaks down default rate by categorical demographic
features.

**Insight to write in your report:** Note which education/marital-status group
shows a higher proportion of defaulters — useful for discussing which customer
segments carry more risk.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(x="EDUCATION", y="default", data=df, ax=axes[0], palette="Blues_d")
axes[0].set_title("Default Rate by Education Level")
axes[0].set_xlabel("Education (1=Grad School, 2=University, 3=High School, 4=Other)")
axes[0].set_ylabel("Default Rate")

sns.barplot(x="MARRIAGE", y="default", data=df, ax=axes[1], palette="Greens_d")
axes[1].set_title("Default Rate by Marital Status")
axes[1].set_xlabel("Marriage (1=Married, 2=Single, 3=Other)")
axes[1].set_ylabel("Default Rate")

plt.tight_layout()
plt.show()

## Block 12: EDA #6 (Bonus) — Repayment Status vs Default

**What this block does:** Examines `PAY_0` (most recent repayment status) against
default rate — this is usually the single strongest predictor in this dataset.

**Insight to write in your report:** Customers with a higher `PAY_0` value
(meaning more months of delayed payment) show a sharply increasing default
rate — a clear, intuitive, and strong relationship.

In [ ]:
plt.figure()
sns.barplot(x="PAY_0", y="default", data=df, palette="Reds_d")
plt.title("Default Rate by Most Recent Repayment Status (PAY_0)")
plt.xlabel("PAY_0 (-1=Paid duly, 1+ = months delayed)")
plt.ylabel("Default Rate")
plt.show()

## Block 13: Feature Scaling Setup — Separate Features and Target

**What this block does:** Splits the DataFrame into `X` (input features) and
`y` (the target we want to predict). This is a required step before any model
training.

In [ ]:
X = df.drop(columns=["default"])
y = df["default"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

## Block 14: Train-Test Split

**What this block does:** Splits the data into a **training set (80%)** and a
**test set (20%)**. The model learns only from the training set and is evaluated
on the unseen test set, which tells us how well it generalizes to new customers.

**Why `stratify=y`:** Because our target is imbalanced (Block 7), stratifying
ensures both the train and test sets keep the same proportion of defaulters —
otherwise the test set could end up with almost no default examples, giving
misleading evaluation results.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Convert y_train and y_test to numeric to allow mean calculation
y_train = pd.to_numeric(y_train)
y_test = pd.to_numeric(y_test)

print("Training set shape:", X_train.shape)
print("Test set shape:", X_test.shape)
print("\nDefault rate in training set:", y_train.mean().round(3))
print("Default rate in test set:", y_test.mean().round(3))

## Block 15: Feature Scaling

**What this block does:** Standardizes all features to have mean 0 and standard
deviation 1 using `StandardScaler`.

**Why this matters:** Features like `LIMIT_BAL` (thousands) and `AGE` (tens) are
on very different scales. Logistic Regression is distance/gradient-based and is
sensitive to this — without scaling, large-scale features would unfairly dominate
the model. We fit the scaler **only on training data** to avoid data leakage, then
apply the same transformation to the test set.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete. Example of scaled training data:")
pd.DataFrame(X_train_scaled, columns=X.columns).head()

## Block 16: Model 1 — Logistic Regression

**What this block does:** Trains a Logistic Regression model — a simple,
interpretable baseline that's standard practice in real-world credit risk
scoring. We use `class_weight="balanced"` to help the model account for the
imbalance in default vs non-default customers, then evaluate it on the test
set with Accuracy, Precision, Recall, and F1-score.

In [ ]:
log_reg = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)

acc_lr = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print("Logistic Regression Results")
print("Accuracy :", round(acc_lr, 4))
print("Precision:", round(prec_lr, 4))
print("Recall   :", round(rec_lr, 4))
print("F1-score :", round(f1_lr, 4))
print("\n", classification_report(y_test, y_pred_lr))

## Block 17: Model 2 — Decision Tree

**What this block does:** Trains a Decision Tree classifier, which can capture
non-linear decision rules (e.g., "if PAY_0 > 1 AND LIMIT_BAL < X, then high
risk"). We limit `max_depth` to prevent overfitting, a point you should be ready
to explain in your viva.

In [ ]:
dt = DecisionTreeClassifier(max_depth=6, class_weight="balanced", random_state=42)
dt.fit(X_train, y_train)   # tree-based models don't require scaled features

y_pred_dt = dt.predict(X_test)

acc_dt = accuracy_score(y_test, y_pred_dt)
prec_dt = precision_score(y_test, y_pred_dt)
rec_dt = recall_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)

print("Decision Tree Results")
print("Accuracy :", round(acc_dt, 4))
print("Precision:", round(prec_dt, 4))
print("Recall   :", round(rec_dt, 4))
print("F1-score :", round(f1_dt, 4))
print("\n", classification_report(y_test, y_pred_dt))

## Block 18: Model 3 — Random Forest

**What this block does:** Trains a Random Forest — an ensemble of many decision
trees — which usually gives the strongest, most stable performance of the three
by reducing overfitting compared to a single tree.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
prec_rf = precision_score(y_test, y_pred_rf)
rec_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

print("Random Forest Results")
print("Accuracy :", round(acc_rf, 4))
print("Precision:", round(prec_rf, 4))
print("Recall   :", round(rec_rf, 4))
print("F1-score :", round(f1_rf, 4))
print("\n", classification_report(y_test, y_pred_rf))

## Block 18b: Model 4 — Gradient Boosting

**What this block does:** Trains a Gradient Boosting Classifier — a boosting
ensemble that builds trees sequentially, where each new tree corrects the
errors of the previous ones. Unlike Random Forest (which builds trees in
parallel/bagging), boosting often improves precision/recall trade-offs on
imbalanced tabular data like this one.

**Why add this model:** Logistic Regression, Decision Tree, and Random Forest
cover a linear model, a single tree, and a bagging ensemble. Gradient
Boosting completes the picture with a boosting ensemble, giving a well-rounded
comparison across four fundamentally different modeling approaches.


In [ ]:
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                 learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)   # tree-based models don't require scaled features

y_pred_gb = gb.predict(X_test)

acc_gb = accuracy_score(y_test, y_pred_gb)
prec_gb = precision_score(y_test, y_pred_gb)
rec_gb = recall_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)

print("Gradient Boosting Performance:")
print(f"  Accuracy : {acc_gb:.4f}")
print(f"  Precision: {prec_gb:.4f}")
print(f"  Recall   : {rec_gb:.4f}")
print(f"  F1-Score : {f1_gb:.4f}")
print("\n", classification_report(y_test, y_pred_gb))


## Block 19: Model Comparison Table

**What this block does:** Builds the required side-by-side comparison table of
all three models across all four metrics, exactly as required in Section 10 of
the project brief.

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Random Forest", "Gradient Boosting"],
    "Accuracy":  [acc_lr, acc_dt, acc_rf, acc_gb],
    "Precision": [prec_lr, prec_dt, prec_rf, prec_gb],
    "Recall":    [rec_lr, rec_dt, rec_rf, rec_gb],
    "F1-Score":  [f1_lr, f1_dt, f1_rf, f1_gb],
})

results = results.round(4)
results


## Block 20: Visual Model Comparison

**What this block does:** Plots all four metrics for all three models side by
side, so the comparison is easy to present on a slide.

In [ ]:
results_melted = results.melt(id_vars="Model", var_name="Metric", value_name="Score")

plt.figure(figsize=(10, 6))
sns.barplot(x="Metric", y="Score", hue="Model", data=results_melted, palette="Set2")
plt.title("Model Comparison Across Evaluation Metrics")
plt.ylim(0, 1)
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## Block 21: Confusion Matrices

**What this block does:** Plots the confusion matrix for each model side by
side. This shows exactly how many actual defaulters were correctly caught
(True Positives) vs missed (False Negatives) — the numbers behind Recall,
which we've argued is the most important metric for this problem.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
models_preds = [("Logistic Regression", y_pred_lr),
                ("Decision Tree", y_pred_dt),
                ("Random Forest", y_pred_rf),
                ("Gradient Boosting", y_pred_gb)]

for ax, (name, preds) in zip(axes, models_preds):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["No Default", "Default"],
                yticklabels=["No Default", "Default"])
    ax.set_title(name)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


## Block 22: ROC Curve Comparison

**What this block does:** Plots the ROC curve and AUC (Area Under Curve) for
each model — a standard way to visually compare classifiers regardless of the
chosen decision threshold. A curve closer to the top-left corner (and a higher
AUC) means a better model.

In [ ]:
plt.figure(figsize=(7, 6))

for name, model, use_scaled in [
    ("Logistic Regression", log_reg, True),
    ("Decision Tree", dt, False),
    ("Random Forest", rf, False),
    ("Gradient Boosting", gb, False),
]:
    X_eval = X_test_scaled if use_scaled else X_test
    probs = model.predict_proba(X_eval)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()


## Block 23: Feature Importance (Random Forest)

**What this block does:** Shows which features the best-performing model
(Random Forest) relied on most heavily. This is valuable for your report's
"Results and Discussion" section and is a very common viva question
("What is the most important feature in your model?").

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 8))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index, palette="viridis")
plt.title("Top 10 Most Important Features (Random Forest)")
plt.xlabel("Importance Score")
plt.show()

print(importances.head(10))

## Block 24: Best Model Selection

**What this block does:** Automatically identifies the best model based on
Recall (since, as discussed, missing an actual defaulter is the costlier
mistake for a bank) and prints a short justification you can paste directly
into your report's conclusion.

In [ ]:
best_row = results.loc[results["Recall"].idxmax()]

print(f"Best performing model (by Recall): {best_row['Model']}")
print(f"  Accuracy : {best_row['Accuracy']}")
print(f"  Precision: {best_row['Precision']}")
print(f"  Recall   : {best_row['Recall']}")
print(f"  F1-Score : {best_row['F1-Score']}")

print("""
Justification: For customer default prediction, Recall is prioritized because
failing to identify an actual defaulter (a False Negative) is more costly for
a bank than incorrectly flagging a reliable customer as risky (a False
Positive). The selected model catches the highest proportion of true
defaulters while maintaining reasonable overall accuracy and precision.
""")

## Block 25 (Optional): Save the Trained Model

**What this block does:** Saves the trained Random Forest model and the
scaler to disk using `joblib`, so it can be reloaded later without retraining
— useful if you want to demo live predictions during your viva.

In [ ]:
import joblib

joblib.dump(rf, "random_forest_default_model.pkl")
joblib.dump(scaler, "feature_scaler.pkl")

print("Model and scaler saved successfully.")

## Conclusion & Future Scope

**Summary:** This project built a customer default prediction system using the
UCI "Default of Credit Card Clients" dataset. After cleaning invalid category
codes, handling class imbalance, and scaling features, four models — Logistic
Regression, Decision Tree, Random Forest, and Gradient Boosting — were trained
and compared. The best-performing model (selected by Recall) is identified in
Block 24 above, and the most influential predictor of default is the
customer's most recent repayment status (`PAY_0`), followed by credit limit
and bill amounts.

**Limitations:**
- The dataset is from 2005 Taiwan and may not generalize to other regions/eras.
- Class imbalance means even a strong model will have some false positives/negatives.
- The model doesn't include macroeconomic context (e.g., interest rates, employment trends).

**Future Scope:**
- Try further boosting algorithms (XGBoost, LightGBM) for potentially higher performance.
- Apply SMOTE or other resampling techniques to address class imbalance more directly.
- Perform hyperparameter tuning (GridSearchCV/RandomizedSearchCV) for each model, including the new Gradient Boosting model.
- Deploy the model as a simple web app (Streamlit/Flask) for live predictions.
